# Technical Indicators & Strategy Backtesting with Alpaca

Run this notebook after installing `requirements.txt` and creating `.env` with Alpaca credentials.


In [ ]:
# After setting up .env, run this cell in Jupyter
from dotenv import load_dotenv
load_dotenv()

from src.data import download_ohlcv
from src.indicators import add_indicators
from src.strategies import buy_and_hold, trend_following, mean_reversion, custom_strategy
from src.backtester import Backtester
from src.metrics import metrics_table
from src.plots import plot_price_signals, plot_equity_curves, plot_drawdowns
from src.report import build_report
from pathlib import Path

ticker = "AAPL"   # change to MSFT, SPY, QQQ, NVDA, etc.
years = 5
initial_capital = 100000

raw = download_ohlcv(ticker, years=years)
data = add_indicators(raw)

bt = Backtester(data, initial_capital=initial_capital)
results = [
    bt.run("Buy & Hold", buy_and_hold(data)),
    bt.run("Trend Following", trend_following(data)),
    bt.run("Mean Reversion", mean_reversion(data)),
    bt.run("Custom Strategy", custom_strategy(data)),
]

metrics = metrics_table(results)
metrics


In [ ]:
output_dir = Path("charts") / ticker
report_dir = Path("reports")
report_dir.mkdir(exist_ok=True)

chart_paths = []
for result in results[1:]:
    chart_paths.append(plot_price_signals(data, result, output_dir))
chart_paths.append(plot_equity_curves(results, output_dir))
chart_paths.append(plot_drawdowns(results, output_dir))

metrics.to_csv(report_dir / f"{ticker}_performance_metrics.csv", index=False)
report_path = build_report(ticker, metrics, chart_paths, report_dir / f"{ticker}_final_report.pdf")
report_path
